<a href="https://colab.research.google.com/github/Poojalakshmi271/RAG-PIPELINE/blob/main/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q pypdf faiss-cpu sentence-transformers
!pip install -q transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 58.4 MB/s eta 0:00:00


In [5]:
from google.colab import files

uploaded = files.upload()

Saving Aptitude Important Topics.pdf to Aptitude Important Topics.pdf


In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Pages Loaded:", len(documents))

/tmp/ipykernel_3951/1797010655.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages Loaded: 2


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 6


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS Vector Store Created")

FAISS Vector Store Created


In [21]:
query = "What is the main topic discussed in the document?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(doc.page_content)


Result 1
SRI ESHWAR COLLEGE OF ENGINEERING 
Department of CSE(Artificial Intelligence and Machine Learning) 
 
 
1. Quantitative Aptitude – Basics & Most Important Topics 
 
Priority Topic What it tests Typical Question Types Approx. Weightage 
★★★★★ Percentages Fast calculations Increase/decrease, successive %, profit-
loss % Very High 
★★★★★ Profit & Loss, Discounts Business math CP, SP, marked price, successive 
discounts Very High

Result 2
2. Logical Reasoning – Basics & Important Topics 
 
Priority Topic Common Question Types 
★★★★★ Seating Arrangement Linear, circular, floor-based, with variables 
★★★★★ Puzzles Scheduling, comparison, grouping, floor puzzles 
★★★★ Syllogism Statements & conclusions (2–3 statements) 
★★★★ Blood Relations Family tree, coded relations 
★★★★ Coding-Decoding Letter shifting, fictitious language 
★★★ Direction Sense Turns, final direction & distance

Result 3
discounts Very High 
★★★★★ Ratio & Proportion Comparison & scaling Partnership, ages, mixtur

In [36]:
from transformers import pipeline
from langchain_huggingface.llms import HuggingFacePipeline # Corrected import path

pipe = pipeline(
    "text2text-generation", # Changed from text-generation to text2text-generation for T5 model
    model="google/flan-t5-base",
    max_new_tokens=256,
    return_full_text=False # Essential for text2text-generation to avoid echoing prompt
)

llm = HuggingFacePipeline(pipeline=pipe)

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [35]:
query = "Summarize the document"

retriever = vectorstore.as_retriever()
docs = retriever.invoke(query)

context = "\n".join([doc.page_content for doc in docs])

# Modified prompt for better summarization with FLAN-T5
prompt = f"Summarize the following text:\n{context}\n\nSummary:"

response = llm.invoke(prompt)

print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarize the following text:
★★★★ Para Jumbles Rearrange 4–5 sentences 
★★★★ Sentence Completion / Fill in blanks Vocabulary + grammar 
★★★ Synonyms & Antonyms Direct word meaning 
★★★ Error Spotting Grammar errors in sentence 
★★★ Sentence Improvement Replace phrase with better one 
★★ Cloze Test Fill blanks in passage 
★★ Critical Reasoning Assumption, inference, strengthening/weakening
★★★ Number/Alphabet Series Missing term, wrong term 
★★★ Inequalities Coded/direct inequalities 
★★ Data Sufficiency 2–3 statements — which are enough? 
★★ Input-Output Machine shifting pattern 
★★ Logical Venn Diagrams Logical relations between groups 
 
 
3. Verbal Ability – Basics & Important Topics 
 
Priority Topic Common Question Types 
★★★★★ Reading Comprehension 1–2 passages (400–600 words), 4–6 questions 
★★★★ Para Jumbles Rearrange 4–5 sentences
SRI ESHWAR COLLEGE OF ENGINEERING 
Department of CSE(Artificial Intelligence and Machine Learning) 
 
 
1. Quantitative Aptitude – Basics & Most Im